In [ ]:
import os

os.environ["DEEPSEEK_API_KEY"] = "sk-135fe459060d4443ab30b8ae1f68f900"
os.environ["TAVILY_API_KEY"] = "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"

In [3]:
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [27]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

instruction = """xml

<role>
你是一个求职公司发现助手，专注于根据用户的求职需求 <query> 结合工具搜寻符合用户需求的公司，并提供这些公司的基本信息。
</role>

<query>
{query}
</query>

<format>
- 请注意，用户的需求可能会有多种表达方式，请确保理解用户的意图
- 请注意现在的时间 {time}，以确保信息的时效性
- 请使用网路搜索工具搜寻符合用户需求的公司
- 请使用表格格式输出公司信息，表格的列名为：公司名称、公司简介、公司官网、公司地址、公司规模、行业
- 参考资料中的采用了的材料如果有网页链接，在末尾以markdown格式输出引用网址的来源：[内容描述](url链接)，必须保证来自于参考资料
</format>
"""

In [28]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate(
    [
        (
            "system",
            instruction,
        ),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{query}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

In [29]:
from langchain.agents import create_react_agent, create_tool_calling_agent
from langchain.agents import AgentExecutor

agent = create_tool_calling_agent(
    llm=llm,
    tools=[search_tool],
    prompt=prompt,
)

In [37]:
import datetime


datetime.datetime.now().isoformat()

'2025-06-30T16:44:52.008847'

In [39]:
import datetime

# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=[search_tool], verbose=True)

result = agent_executor.invoke(
    {
        "query": "我是2026届计算机专业的毕业生，想找到开放校招的中国公司，最好是互联网行业的。请帮我搜索一下相关信息。",
        "time": datetime.datetime.now().isoformat(),
    }
)



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': '2026届校招 中国互联网公司', 'search_depth': 'advanced'}`


{'query': '2026届校招 中国互联网公司', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://github.com/namewyf/Campus2026', 'title': 'namewyf/Campus2026: 2026届互联网校招&2025互联网实习信息汇总', 'content': '| 公司 | 招聘状态&&投递链接 | 更新日期 | 地点 | 备注 |\n| --- | --- | --- | --- | --- |\n| 腾讯 | 暑期实习&日常实习 | 2024/12/27 | 全国 | 推文，全年可投递，越早投越好 |\n| 字节跳动 | 寒假青训营 | 2024/12/27 | 全国 | 门槛低，完成15道算法题即可参加 |\n| 阿里巴巴 | 暑期实习 | 2025/3/8 | 全国 | 已开启投递 |\n| 携程旅行 | 暑期实习 | 2025/3/20 | 全国 | 内推码：NTAciAS |\n| 百度 | 暑期实习 | 2025/3/2 | 全国 | 已开启投递 |\n| 美团 | 暑期实习 | 2025/3/5 | 全国 | 已开启投递 |\n| 京东 | 暑期实习 | 2025/3/3 | 全国 | 已开启投递 |\n| 网易 | 雷火暑期实习 | 2025/3/8 | 杭州 | 游戏部门已开启 |\n| 华为 | 暑期实习 | 2025/3/2 | 全国 | 已开启投递 |\n| 小米 | 暑期实习 | 2025/3/8 | 全国 | 已开启投递 | [...] | 公司 | 招聘状态&&投递链接 | 更新日期 | 地点 | 备注 |\n| --- | --- | --- | --- | --- |\n| 北森 | 暑期实习 | 2025/3/7 |  |  |\n| 金山软件 | 日常实习 | 2025/3/7 |  |  |\

In [26]:
result

{'query': '小鹏汽车现在有校招岗位吗？',
 'output': '根据搜索结果，小鹏汽车目前有校招岗位，以下是相关信息：\n\n| 公司名称 | 公司简介 | 公司官网 | 公司地址 | 公司规模 | 行业 |\n|----------|----------|----------|----------|----------|------|\n| 小鹏汽车 | 小鹏汽车是中国领先的智能电动汽车制造商，致力于通过科技驱动未来出行。 | [campus.xiaopeng.com](https://campus.xiaopeng.com/) | 中国广州 | 5000人以上 | 智能电动汽车 |\n\n### 参考资料\n1. [小鹏汽车校招官网](https://campus.xiaopeng.com/)\n2. [小鹏汽车2023年校招信息](https://xps01.xiaopeng.com/cms/material/other/2025/04-17/other_20250417183716_92569.pdf)'}